# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

In [41]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

In [42]:
#reading the vehicles dataset and printing the first five rows
df = pd.read_csv('data/vehicles.csv')

In [43]:
#print first five rows of the data
df.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


In [53]:
#print the info of the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 393985 entries, 0 to 426879
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   region          393985 non-null  object 
 1   price           393985 non-null  int64  
 2   year            392812 non-null  Int64  
 3   manufacturer    377800 non-null  object 
 4   model           389284 non-null  object 
 5   condition       242596 non-null  object 
 6   cylinders       233575 non-null  object 
 7   fuel            391391 non-null  object 
 8   odometer        391695 non-null  float64
 9   title_status    386251 non-null  object 
 10  transmission    392162 non-null  object 
 11  drive           273731 non-null  object 
 12  size            111052 non-null  object 
 13  type            308053 non-null  object 
 14  paint_color     276836 non-null  object 
 15  state           393985 non-null  object 
 16  age             392812 non-null  Int64  
 17  miles_per_year 

In [45]:
#lets describe the dataset
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,426880.0,NaN,NaN,NaN,7311486634.224333,4473170.412559,7207408119.0,7308143339.25,7312620821.0,7315253543.5,7317101084.0
region,426880,404,columbus,3608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,426880.0,NaN,NaN,NaN,75199.033187,12182282.173598,0.0,5900.0,13950.0,26485.75,3736928711.0
year,425675.0,NaN,NaN,NaN,2011.235191,9.45212,1900.0,2008.0,2013.0,2017.0,2022.0
manufacturer,409234,42,ford,70985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,421603,29649,f-150,8009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
condition,252776,6,good,121456,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cylinders,249202,8,6 cylinders,94169,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fuel,423867,5,gas,356209,NaN,NaN,NaN,NaN,NaN,NaN,NaN
odometer,422480.0,NaN,NaN,NaN,98043.331443,213881.500798,0.0,37704.0,85548.0,133542.5,10000000.0


In [46]:
#Let's try to find some unique values in some columns
print("Unique values in 'condition' column:", df['condition'].unique())
print("Unique values in 'title_status' column:", df['title_status'].unique())
print("Unique values in 'fuel' column:", df['fuel'].unique())
print("Unique values in 'size' column:", df['size'].unique())
print("Unique values in 'drive' column:", df['drive'].unique())
print("Unique values in 'type' column:", df['type'].unique())

Unique values in 'condition' column: [nan 'good' 'excellent' 'fair' 'like new' 'new' 'salvage']
Unique values in 'title_status' column: [nan 'clean' 'rebuilt' 'lien' 'salvage' 'missing' 'parts only']
Unique values in 'fuel' column: [nan 'gas' 'other' 'diesel' 'hybrid' 'electric']
Unique values in 'size' column: [nan 'full-size' 'mid-size' 'compact' 'sub-compact']
Unique values in 'drive' column: [nan 'rwd' '4wd' 'fwd']
Unique values in 'type' column: [nan 'pickup' 'truck' 'other' 'coupe' 'SUV' 'hatchback' 'mini-van' 'sedan'
 'offroad' 'bus' 'van' 'convertible' 'wagon']


### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

1. Drop/clean identifiers

In [52]:
#dropping Id and VIN columns as they are not useful for our analysis
df = df.drop(columns=['id', 'VIN'])

KeyError: "['id', 'VIN'] not found in axis"

2. Fix datatypes and simple cleaning

In [54]:
#As we can see year is float let's convert to int if safe
df['year'] = df['year'].astype('Int64')
df['odometer'] = df['odometer'].astype(float)

3. Handle missing values & Outlier treatment

In [58]:
#Remove zero or negative prices (if any)
df = df[df['price'] > 0]
df['log_price'] = np.log1p(df['price'])

4. Feature Engineering

In [56]:
#Let's identify Age of each vehicle and create a new column 'age'
current_year = 2025
df['age'] = current_year - df['year']

In [57]:
#Add another feature for mileage per year
df['miles_per_year'] = df['odometer'] / df['age'].replace(0, np.nan)  # avoid division by zero

#Add feature for is_automatic based on transmission type
df['is_automatic'] = df['transmission'].str.lower().str.contains('auto', case=False, na=False).astype(int)

#Add feature for is salvaged based on title_status
df['is_salvage'] = df['title_status'].str.lower().str.contains('salvage|rebuilt', case=False, na=False).astype(int)

df.head()

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state,age,miles_per_year,is_automatic,is_salvage
0,prescott,6000,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,<NA>,<NA>,0,0
1,fayetteville,11900,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,<NA>,<NA>,0,0
2,florida keys,21000,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,<NA>,<NA>,0,0
3,worcester / central MA,1500,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,<NA>,<NA>,0,0
4,greensboro,4900,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,<NA>,<NA>,0,0


5. Encoding categorical variables

Different strategies for different columns:

Low-cardinality (e.g., condition, fuel, transmission, drive, size, type, title_status, paint_color if not too many): One-hot encoding.

High-cardinality (e.g., model, sometimes manufacturer): frequency encoding or target encoding

In [59]:
# Factorize — gives integer labels and mapping
df['manufacturer_code'], manuf_mapping = pd.factorize(df['manufacturer'])
# or
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['condition_code'] = le.fit_transform(df['condition'].astype(str))

# Frequency encoding
freq = df['model'].value_counts(normalize=True)
df['model_freq'] = df['model'].map(freq)

# Target encoding (example using mean price) -- do inside Cross Validation (CV) to avoid leakage
mean_price_by_model = df.groupby('model')['price'].mean()
df['model_target_mean'] = df['model'].map(mean_price_by_model)

6. Final feature list

In [60]:
features = [
    'age','odometer','miles_per_year','manufacturer_code','model_freq',
    'condition_code','cylinders','fuel','is_automatic','is_salvage','drive','size','type','paint_color','state'
]

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.